In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
website = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE169nnn/GSE169598/matrix/GSE169598_series_matrix.txt.gz"
data_path = "fajciarky_data/data.txt"
data_path_zip = "fajciarky_data/data.txt.gz"

In [3]:
# !curl $website --create-dirs -o $data_path_zip
# !gunzip $data_path_zip

In [3]:
skiprow = None

with open(data_path, "r") as f:

    for i, line in enumerate(f):

        if line.startswith("\"ID_REF\""):
            skiprow = i
            break

print("Table starts at row:", skiprow)

Table starts at row: 64


In [4]:
df = pd.read_csv(
    data_path,
    sep="\t",
    comment="!",
    header=0,
    skiprows=skiprow
)

df.set_index("ID_REF", inplace=True)

print("\nOriginal shape:")
print(df.shape)


Original shape:
(714666, 96)


In [5]:
print("\nFirst rows:")
print(df.head())



First rows:
            GSM5210464  GSM5210465  GSM5210466  GSM5210467  GSM5210468  \
ID_REF                                                                   
cg00000029    0.006342    0.101510    0.042916    0.101567    0.114736   
cg00000109    0.911428    0.924547    0.926018    0.920207    0.925702   
cg00000155    0.948963    0.936607    0.946371    0.951434    0.949158   
cg00000158    0.955034    0.954614    0.964843    0.958018    0.954761   
cg00000165    0.602739    0.654239    0.635608    0.514626    0.569226   

            GSM5210469  GSM5210470  GSM5210471  GSM5210472  GSM5210473  ...  \
ID_REF                                                                  ...   
cg00000029    0.129332    0.134621    0.105126    0.116374    0.115263  ...   
cg00000109    0.920111    0.928904    0.930301    0.936867    0.929511  ...   
cg00000155    0.941937    0.940608    0.952269    0.940826    0.945244  ...   
cg00000158    0.952986    0.948309    0.949837    0.964955    0.958335  .

In [6]:
df = df.astype(float)

df = df.T

print("\nTransposed shape:")
print(df.shape)


Transposed shape:
(96, 714666)


In [7]:
metadata = {}

with open(data_path, "r") as file:

    characteristic_counter = 0

    for line in file:
        if line.startswith("!Sample_title"):

            metadata["title"] = line.strip().split("\t")[1:]
        elif line.startswith("!Sample_geo_accession"):

            metadata["accession"] = line.strip().split("\t")[1:]
        elif line.startswith("!Sample_characteristics_ch1"):

            values = line.strip().split("\t")[1:]
            
            if characteristic_counter == 0:
                metadata["sex"] = values
            elif characteristic_counter == 1:
                metadata["gestational_age"] = values
            elif characteristic_counter == 2:
                metadata["smoke_type"] = values

            characteristic_counter += 1
        elif line.startswith("!series_matrix_table_begin"):
            break

meta_df = pd.DataFrame(metadata)
# meta_df = meta_df.applymap(lambda x: x.replace('"', ''))
meta_df = meta_df.apply(lambda s: s.str.replace('"', '', regex=False))
print(meta_df.columns)

Index(['title', 'accession', 'sex', 'gestational_age', 'smoke_type'], dtype='str')


In [8]:
print(meta_df.head())


             title   accession     sex              gestational_age  \
0  placental DNA 1  GSM5210464  Sex: F  gestational age weeks: 39.4   
1  placental DNA 2  GSM5210465  Sex: F  gestational age weeks: 39.3   
2  placental DNA 3  GSM5210466  Sex: F  gestational age weeks: 39.3   
3  placental DNA 4  GSM5210467  Sex: M  gestational age weeks: 39.7   
4  placental DNA 5  GSM5210468  Sex: F  gestational age weeks: 37.7   

           smoke_type  
0  smoking: nonsmoker  
1     smoking: smoker  
2  smoking: nonsmoker  
3     smoking: smoker  
4     smoking: smoker  


In [9]:
def create_label(smoke_value):

    value = smoke_value.lower()

    if "nonsmoker" in value:
        return "NonSmoker"

    elif "smoker" in value:
        return "Smoker"

    else:
        return "Unknown"
    
meta_df["label"] = meta_df["smoke_type"].apply(create_label)

print("\nLabel counts:")
print(meta_df["label"].value_counts())


Label counts:
label
Smoker       72
NonSmoker    24
Name: count, dtype: int64


In [10]:
meta_df = meta_df.set_index("accession")

y = meta_df.loc[df.index, "label"]

X = df.copy()

print(X.shape)
print(y.value_counts())

(96, 714666)
label
Smoker       72
NonSmoker    24
Name: count, dtype: int64


In [11]:
nan_frac = X.isna().mean(axis=0)
# print how many probes have any missingness
print(f'any missingness: {(nan_frac > 0).sum():,}')

any missingness: 0


In [15]:
meta_df

,title,sex,gestational_age,smoke_type,label
accession,,,,,
GSM5210464,placental DNA 1,Sex: F,gestational age weeks: 39.4,smoking: nonsmoker,NonSmoker
GSM5210465,placental DNA 2,Sex: F,gestational age weeks: 39.3,smoking: smoker,Smoker
GSM5210466,placental DNA 3,Sex: F,gestational age weeks: 39.3,smoking: nonsmoker,NonSmoker
GSM5210467,placental DNA 4,Sex: M,gestational age weeks: 39.7,smoking: smoker,Smoker
GSM5210468,placental DNA 5,Sex: F,gestational age weeks: 37.7,smoking: smoker,Smoker
...,...,...,...,...,...
GSM5210564,placental DNA 92,Sex: F,gestational age weeks: 37.0,smoking: smoker,Smoker
GSM5210565,placental DNA 93,Sex: F,gestational age weeks: 37.9,smoking: smoker,Smoker
GSM5210566,placental DNA 94,Sex: F,gestational age weeks: 39.0,smoking: smoker,Smoker


In [20]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
meta_df_train = meta_df.loc[X_train.index]
meta_df_test = meta_df.loc[X_test.index]

In [16]:
variances = X.var(axis=0)

In [ ]:
# 1. compute variances ONLY on training data
variances = X_train.var(axis=0)

# 2. select top-k features based on training set only
top_k = 3_000
top_features = variances.sort_values(ascending=False).head(top_k).index

# 3. filter both train and test using SAME features
X_train_filtered = X_train[top_features]
X_test_filtered = X_test[top_features]

print("\nAfter feature filtering:")
print("Train shape:", X_train_filtered.shape)
print("Test shape:", X_test_filtered.shape)


After feature filtering:
(96, 3000)


In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)

sfs = SequentialFeatureSelector(
    model,
    n_features_to_select=50,
    direction="forward",
    scoring="accuracy",
    cv=3 
)


sfs.fit(X_train_filtered, y_train)

X_train_sequential = sfs.transform(X_train_filtered)
X_test_sequential = sfs.transform(X_test_filtered)


print("Selected shape:", X_train_sequential.shape)

Selected shape: (96, 50)


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=100)
X_train_kbest = selector.fit_transform(X_train_filtered, y_train)
X_test_kbest = selector.transform(X_test_filtered)


print("KBest shape:", X_train_kbest.shape)

KBest shape: (96, 100)
